# LC7 — Cleaning and storing energy time series (self-paced, ~45 min)

You have been handed a year of hourly Svedala data — exported carelessly, as real data usually is. By the end of this notebook it is clean, every repair is documented, and it lives in analytical storage. Material from this notebook appears in **Quiz 2**, and Lab 5 uses exactly this working method.

**The one rule of cleaning:** there is a difference between *fixing an error* and *inventing a measurement*. Every repair below gets a line in a repairs log — the habit that becomes your project's decision log.

In [ ]:
# Install exactly what this notebook uses. numpy comes with pandas but we
# name it: explicit beats implicit, in dependencies too.
%pip install pandas numpy pyarrow duckdb matplotlib --quiet

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../data/svedala-year/svedala_hourly_dirty.csv",
                  index_col=0, parse_dates=True)
repairs = []            # every decision lands here
raw.info()

## 1. Look before you touch

8780 rows in a year that has 8760 hours — already suspicious. Three quick views: the index, the statistics, the picture.

In [ ]:
print("duplicated timestamps:", raw.index.duplicated().sum())
print("index monotonic:", raw.index.is_monotonic_increasing)
raw.describe().round(1)

In [ ]:
axes = raw[[c for c in raw.columns if c.startswith("ZON_")]].plot(
    subplots=True, figsize=(10, 6), title="Before cleaning — spot the problems");

Negative load, a spike five times anything plausible, and a zone that drops to almost zero for a day — `describe()` and the plot found in seconds what no amount of staring at the CSV would.

## 2. Timestamps first: local time is a trap

The exporter wrote **naive local timestamps** (Europe/Stockholm). Twice a year that goes wrong: the spring DST switch has a missing hour, the autumn switch has the same wall-clock hour **twice**. Repair = localize honestly, convert to UTC, and work in UTC forever after.

In [ ]:
df = raw.copy()
idx = df.index.tz_localize("Europe/Stockholm", ambiguous="infer", nonexistent="shift_forward")
df.index = idx.tz_convert("UTC")
df.index.name = "timestamp"          # it is not local anymore — say so
df = df.sort_index()
repairs.append("Localized naive Europe/Stockholm timestamps (DST handled), converted to UTC.")
print("autumn duplicate hour resolved:", df.index.tz)

In [ ]:
dups = df.index.duplicated(keep="first").sum()
df = df[~df.index.duplicated(keep="first")]
repairs.append(f"Dropped {dups} exactly duplicated rows (kept first occurrence).")
print(len(df), "rows after de-duplication")

## 3. The unit glitch

One day of ZON_NORR sits near 0.6 MW in a zone that averages ~600. That is not an outage — it is **a unit error** (GW written as MW). The give-away: the *shape* of the day is perfectly normal, only the magnitude is off by exactly 1000.

In [ ]:
# Only POSITIVE values can be a GW-as-MW error (they come out ~1000x too
# small). A negative reading is a different fault - leave it to the
# impossible-value step below, or this repair turns -470.7 into -470700.
suspect = (df["ZON_NORR"] > 0) & (df["ZON_NORR"] < df["ZON_NORR"].median() * 0.05)
print("suspect hours:", suspect.sum(), "| all on:", sorted(set(df.index[suspect].date)))
df.loc[suspect, "ZON_NORR"] *= 1000
repairs.append(f"ZON_NORR {suspect.sum()} h on 2025-03-05: values x1000 (GW-as-MW unit error, shape intact).")

Notice the reasoning in the log entry: we did not just "fix an outlier" — we identified a *mechanism* (unit error) and repaired it losslessly. That is the difference between cleaning and cosmetics.

## 4. Spikes, negatives, gaps

Physically impossible values (negative consumption) and isolated spikes become **missing** — we do not know the true value, so we say so. Then short gaps are interpolated; long gaps stay honest NaN.

In [ ]:
zones = [c for c in df.columns if c.startswith("ZON_")]
# Per zone: flag physically impossible values (negative load) and statistical
# spikes (further from the 49 h rolling median than 4 standard deviations),
# then turn BOTH into NaN — we do not know the true value, so we say so.
for c in zones:
    neg = df[c] < 0
    med = df[c].rolling(49, center=True, min_periods=12).median()
    spike = (df[c] - med).abs() > 4 * df[c].std()
    n = (neg | spike).sum()
    df.loc[neg | spike, c] = np.nan
    if n: repairs.append(f"{c}: {n} impossible/spike values set to NaN.")
gaps_before = df[zones].isna().sum().sum()
df[zones] = df[zones].interpolate(limit=3)             # bridge up to 3 h, no more
repairs.append(f"Zone columns: {gaps_before - df[zones].isna().sum().sum()} of {gaps_before} "
               "missing values interpolated (gaps <= 3 h); longer gaps left as NaN.")
df["total_mw"] = df[zones].sum(axis=1, min_count=len(zones)).round(1)
repairs.append("total_mw recomputed from cleaned zones (NaN where any zone is missing).")
print(df[zones].isna().sum())

In [ ]:
print("temp_mid gap:", df["temp_mid"].isna().sum(), "hours in July — a real sensor outage.")
repairs.append("temp_mid: 72 h outage in July LEFT AS NaN — inventing three days of "
               "temperature would be fabrication, not repair.")

## 5. Sanity checks — as code, not as glances

A cleaned dataset should *prove* it is clean. These asserts are the Lecturecise 5 idea applied to data — and they run in CI exactly like your other tests:

In [ ]:
assert df.index.is_unique and df.index.is_monotonic_increasing
assert len(df) == 8760, f"expected 8760 hours, got {len(df)}"
assert (df[zones].dropna() >= 0).all().all(), "negative load survived cleaning"
assert df[zones].isna().mean().max() < 0.01, "too much missing data remains"
ratio = df["total_mw"].max() / df["total_mw"].mean()
assert 1.1 < ratio < 1.8, f"peak/mean ratio {ratio:.2f} implausible for zonal load"
print("all sanity checks pass")

## 6. Store it properly: Parquet + DuckDB

CSV was the delivery format; it is not an analysis format. **Parquet** keeps types, compresses, loads in milliseconds. **DuckDB** puts SQL on top of it — no server, one file, made for exactly this.

In [ ]:
df.to_parquet("svedala_hourly_clean.parquet")
import duckdb
con = duckdb.connect("svedala.duckdb")
con.execute("CREATE OR REPLACE TABLE hourly AS SELECT * FROM 'svedala_hourly_clean.parquet'")
con.execute("""
    SELECT date_trunc('day', timestamp) AS day,
           max(total_mw) AS peak_mw
    FROM hourly WHERE timestamp >= '2025-01-13' AND timestamp < '2025-01-20'
    GROUP BY day ORDER BY day
""").df()

## 7. Sidebar: two data-quality stories you already know

Cleaning is not only about time series. The Svedala *network* data carries the same lessons: the line current limits (`max_i_ka`) are **empty** in the source — every loading percentage you have ever computed rests on defaults we chose and documented. And look at the transformer ratings:

In [ ]:
t = pd.read_csv("../data/svedala/transformers.csv", index_col=0)
t[["name", "sn_mva", "vn_hv_kv", "vn_lv_kv"]].nsmallest(5, "sn_mva")

100 MVA nameplates on 400 kV step-up transformers that carry hundreds of MW — **placeholder values**, not engineering. This is why the course's security criterion counts *line* loadings only, and why every dataset you publish needs a README that says what can and cannot be trusted. Data quality is not a preprocessing step; it is a property you must establish before any result downstream means anything.

## 8. The repairs log — read it back

This list is the seed of a habit: in Period 2, `DECISIONS.md` plays exactly this role for your whole project.

In [ ]:
for i, r in enumerate(repairs, 1):
    print(f"{i:2d}. {r}")

## Self-check

In [ ]:
ref = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
common = df[zones].dropna()
match = (np.isclose(common, ref.loc[common.index, zones], rtol=0.02)).mean()
assert match > 0.97, f"cleaned data matches reference on only {match:.0%} of cells"
print(f"ALL OK — cleaned data matches the reference on {match:.0%} of cells.")
print("(The rest are the hours you repaired — compare a few and judge your repairs.)")